In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
import open3d as o3d
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, TensorDataset

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:

path_nic = "C:/Users/Nicola/Desktop/Jona" 

#  Alle .las-Dateien aus einem Ordner automatisch finden
def find_las_files(directory):
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".las")]

#  Punktwolken aus allen Dateien einlesen und zusammenführen
def load_and_merge_las(files, chunk_size=100000):
    all_points = []
    for file in tqdm(files, desc="Lade .las-Dateien"):
        las = laspy.read(file)
        total_points = len(las.x)
        for start in range(0, total_points, chunk_size):
            end = min(start + chunk_size, total_points)
            chunk = np.vstack((
                las.x[start:end],
                las.y[start:end],
                las.z[start:end],
                las.classification[start:end],  # Moved classification here
                las.intensity[start:end],
                las.return_number[start:end],
                las.number_of_returns[start:end]
            )).T
            all_points.append(chunk)
    return np.vstack(all_points)


#  KD-Tree für Nachbarschaftssuche erstellen (optional)
def build_kdtree(points):
    return cKDTree(points)

#  PyTorch DataLoader erstellen
def create_dataloader(points, batch_size=64):
    tensor_data = torch.tensor(points, dtype=torch.float32)
    dataset = TensorDataset(tensor_data)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Hauptprogramm

directory = path_nic
las_files = find_las_files(directory)
if not las_files:
    print("Keine .las-Dateien gefunden!")
else:
    print(f"{len(las_files)} .las-Dateien gefunden.")
    points = load_and_merge_las(las_files)



    print(f"Gesamtpunkte: {points.shape}")



9 .las-Dateien gefunden.


Lade .las-Dateien: 100%|██████████| 9/9 [00:11<00:00,  1.25s/it]


Gesamtpunkte: (90186624, 7)


In [ ]:
def model_3d(df, point_size=1, dtype="laz"):
    """
    Creates a 3D point cloud visualization.

    Parameters:
        df: pandas.DataFrame: DataFrame containing point cloud data.
            Must contain columns for x, y, z coordinates and classification labels.
        point_size (float, optional): Size of the points in the 3D visualization.
        dtype (str, optional): Type of data. Can be "las" or "laz". (defines the color scheme)
            Default is "laz".
    """

    if dtype == "las":
        color_map = {1: [0.5, 0.5, 0.5],   
                        2: [0.33, 0.42, 0.18], 
                        3: [0.0, 0.39, 0.0],   
                        6: [0.6, 0.3, 0.0],    
                        9: [0.0, 0.0, 1.0],   
                        17: [0.4, 0.26, 0.13] 
                        }
    elif dtype == "laz": 
        color_map = {
    1: [0.6, 0.6, 0.6],  # Übriges: Grau
    2: [0.2, 0.6, 0.2],  # Bodenpunkte / DOM + DTM: Dunkelgrün
    3: [0.0, 0.8, 0.0],  # Vegetation tief (< .5m) / DOM: Helles Grün
    4: [0.0, 0.6, 0.0],  # Vegetation mittel (< 3m) / DOM: Mittelgrün
    5: [0.0, 0.4, 0.0],  # Vegetation hoch (> 3m) / DOM: Dunkelgrün
    6: [0.6, 0.3, 0.0],  # Gebäude / DOM: Braun
    7: [0.8, 0.0, 0.0],  # Ausreißer und Fehlmessungen: Rot
    10: [0.0, 0.0, 0.6], # Brücken und Stege / DOM: Blau
    12: [0.6, 0.6, 0.0], # Streifenrandpunkte: Gelb
    15: [0.9, 0.6, 0.0], # Freileitungen, Masten und Antennen: Orange
}



    # Spaltennamen in Kleinbuchstaben mappen
    cols = {col.lower(): col for col in df.columns}

    required = ["x", "y", "z", "classification"]
    if not all(k in cols for k in required):
        raise ValueError(f"DataFrame muss die Spalten {required} enthalten (Groß-/Kleinschreibung egal).")

    xyz = df[[cols["x"], cols["y"], cols["z"]]].to_numpy()
    labels = df[cols["classification"]].astype(int).to_numpy()
    colors = np.array([color_map.get(lbl, [1, 1, 1]) for lbl in labels])  # Fallback: weiß

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.colors = o3d.utility.Vector3dVector(colors)

    vis = o3d.visualization.Visualizer()
    vis.create_window()
    vis.add_geometry(pcd)

    opt = vis.get_render_option()
    opt.point_size = point_size

    vis.run()
    vis.destroy_window()